In [1]:
import time
from pathlib import Path

import feedparser
import pandas as pd
from newspaper import Article

In [2]:
# the SAP news feed only shows the latest ~30 posts on page 1,
# so we page back a few times to get more volume
base_url = "https://news.sap.com/feed/"

sap_docs = []

for page in range(1, 6):
    feed_url = base_url if page == 1 else f"{base_url}?paged={page}"
    feed = feedparser.parse(feed_url)

    if not feed.entries:
        break

    for entry in feed.entries:
        sap_docs.append({
            "title": entry.title,
            "url": entry.link,
            "published": entry.published,
            "source": "SAP News",
            "description": ""
        })

    time.sleep(0.3)

len(sap_docs)

150

In [3]:
sap_df = pd.DataFrame(sap_docs).drop_duplicates(subset="url").reset_index(drop=True)

len(sap_df)

150

In [4]:
# scrape the full article text for each entry, skip the ones that fail
contents = []

for url in sap_df["url"]:

    try:

        article = Article(url)

        article.download()
        article.parse()

        contents.append(article.text)

    except:

        contents.append("")

In [5]:
sap_df["content"] = contents

In [6]:
# works whether this notebook runs from its own folder or from repo root (e.g. via main.ipynb)
data_dir = Path("../notebook/data")
if not data_dir.exists():
    data_dir = Path("notebook/data")
data_dir.mkdir(parents=True, exist_ok=True)

sap_df.to_json(data_dir / "sap_news.json", orient="records", indent=2)

print("saved", len(sap_df), "sap articles to", data_dir / "sap_news.json")

saved 150 sap articles to ../notebook/data/sap_news.json
